## Daily volume check (Database1)

**Use this notebook** to see whether the data engineer loaded new data today.

Each run:
1. Counts every source table in Database1
2. Counts, per patient: visits, labs, histories, notes, unique claims (ClaimNo + ClaimLineNo)
3. Saves a snapshot in your analytics DB
4. Diffs this run against the previous snapshot

Edit **CONFIG** only. If the DE used `VisitId` or `MEMBER_ID`, change it there - queries are built from CONFIG.

Upload this notebook to Snowflake, pick a warehouse, and connect the kernel. You do **not** need account/user/password in the notebook - the kernel already has a session.

Run `setup_volume_snapshots.sql` once first (or the setup cell below).


### Why a notebook

| Option | Use when |
|--------|----------|
| **This notebook** | Daily “is there new data?” - config once, tables + patient diffs, easy to re-run |


A one-shot SQL `COUNT(*)` cannot tell you what changed vs yesterday. You need a saved snapshot. The notebook writes that snapshot and prints the delta.


In [ ]:
# ========== CONFIG (edit these only) ==========
CONFIG = {
    # DE source
    "source_db": "DATABASE1",
    "source_schema": "PUBLIC",
    # Your analytics DB (snapshots live here, not in Database1)
    "snapshot_db": "MY_ANALYTICS_DB",
    "snapshot_schema": "STAGING",
    # Dictionary names - change if DE used MemberId / VisitId / PATIENT_ID / etc.
    "patient_id_col": "PatientId",
    "encounter_id_col": "EncounterId",  # VisitId if that is the real column
    "claim_no_col": "ClaimNo",
    "claim_line_col": "ClaimLineNo",
    "tables": {
        "CENSUS": "CENSUS",
        "ENCOUNTER_VISIT": "ENCOUNTER_VISIT",
        "LAB": "LAB",
        "SURGICAL_HISTORY": "SURGICAL_HISTORY",
        "MEDICAL_HISTORY": "MEDICAL_HISTORY",
        "FAMILY_HISTORY": "FAMILY_HISTORY",
        "CLINICAL_NOTE": "CLINICAL_NOTE",
        "CLAIM": "CLAIM",
    },
    "record_id_col": {
        "LAB": "LabId",
        "SURGICAL_HISTORY": "SurgicalHistoryId",
        "MEDICAL_HISTORY": "MedicalHistoryId",
        "FAMILY_HISTORY": "FamilyHistoryId",
        "CLINICAL_NOTE": "NoteId",
    },
}


In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()


def q(col: str) -> str:
    return f'"{col}"'


def fqn(table_key: str) -> str:
    name = CONFIG["tables"][table_key]
    return f'{CONFIG["source_db"]}.{CONFIG["source_schema"]}.{name}'


def snap(name: str) -> str:
    return f'{CONFIG["snapshot_db"]}.{CONFIG["snapshot_schema"]}.{name}'


def claim_key_expr() -> str:
    return f"{q(CONFIG['claim_no_col'])} || '-' || {q(CONFIG['claim_line_col'])}"


print("Using the notebook kernel session.")


In [ ]:
# One-time snapshot tables (safe to re-run)
session.sql(f'''
CREATE TABLE IF NOT EXISTS {snap("VOLUME_CHECK_RUN")} (
    run_id         STRING PRIMARY KEY,
    checked_at     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    source_db      STRING,
    source_schema  STRING
)
''').collect()

session.sql(f'''
CREATE TABLE IF NOT EXISTS {snap("VOLUME_TABLE_SNAPSHOT")} (
    run_id                STRING,
    table_name            STRING,
    row_count             NUMBER,
    distinct_patients     NUMBER,
    distinct_encounters   NUMBER,
    distinct_record_ids   NUMBER,
    PRIMARY KEY (run_id, table_name)
)
''').collect()

session.sql(f'''
CREATE TABLE IF NOT EXISTS {snap("VOLUME_PATIENT_SNAPSHOT")} (
    run_id                    STRING,
    patient_id                STRING,
    visit_count               NUMBER,
    lab_count                 NUMBER,
    surgical_history_count    NUMBER,
    medical_history_count     NUMBER,
    family_history_count      NUMBER,
    note_count                NUMBER,
    claim_count               NUMBER,
    PRIMARY KEY (run_id, patient_id)
)
''').collect()
print("Snapshot tables ready.")


In [ ]:
# 1) Table sizes
p = q(CONFIG["patient_id_col"])
e = q(CONFIG["encounter_id_col"])
ck = claim_key_expr()
ids = CONFIG["record_id_col"]

parts = [
    f'''SELECT 'CENSUS' AS table_name, COUNT(*) AS row_count,
               COUNT(DISTINCT {p}) AS distinct_patients,
               NULL::NUMBER AS distinct_encounters,
               COUNT(DISTINCT {p}) AS distinct_record_ids
        FROM {fqn("CENSUS")}''',
    f'''SELECT 'ENCOUNTER_VISIT', COUNT(*),
               COUNT(DISTINCT {p}), COUNT(DISTINCT {e}), COUNT(DISTINCT {e})
        FROM {fqn("ENCOUNTER_VISIT")}''',
    f'''SELECT 'LAB', COUNT(*),
               COUNT(DISTINCT {p}), COUNT(DISTINCT {e}),
               COUNT(DISTINCT {q(ids["LAB"])})
        FROM {fqn("LAB")}''',
    f'''SELECT 'SURGICAL_HISTORY', COUNT(*),
               COUNT(DISTINCT {p}), COUNT(DISTINCT {e}),
               COUNT(DISTINCT {q(ids["SURGICAL_HISTORY"])})
        FROM {fqn("SURGICAL_HISTORY")}''',
    f'''SELECT 'MEDICAL_HISTORY', COUNT(*),
               COUNT(DISTINCT {p}), COUNT(DISTINCT {e}),
               COUNT(DISTINCT {q(ids["MEDICAL_HISTORY"])})
        FROM {fqn("MEDICAL_HISTORY")}''',
    f'''SELECT 'FAMILY_HISTORY', COUNT(*),
               COUNT(DISTINCT {p}), COUNT(DISTINCT {e}),
               COUNT(DISTINCT {q(ids["FAMILY_HISTORY"])})
        FROM {fqn("FAMILY_HISTORY")}''',
    f'''SELECT 'CLINICAL_NOTE', COUNT(*),
               COUNT(DISTINCT {p}), COUNT(DISTINCT {e}),
               COUNT(DISTINCT {q(ids["CLINICAL_NOTE"])})
        FROM {fqn("CLINICAL_NOTE")}''',
    f'''SELECT 'CLAIM', COUNT(*),
               COUNT(DISTINCT {p}), COUNT(DISTINCT {e}),
               COUNT(DISTINCT {ck})
        FROM {fqn("CLAIM")}''',
]

table_sizes = session.sql(" UNION ALL ".join(parts)).to_pandas()
table_sizes.columns = [c.lower() for c in table_sizes.columns]
display(table_sizes.sort_values("table_name"))


In [ ]:
# 2) Per-patient: visits, labs, histories, notes, unique ClaimNo+ClaimLineNo
patient_sql = f'''
WITH census AS (
    SELECT {p} AS patient_id FROM {fqn("CENSUS")}
),
visits AS (
    SELECT {p} AS patient_id, COUNT(DISTINCT {e}) AS visit_count
    FROM {fqn("ENCOUNTER_VISIT")} GROUP BY 1
),
labs AS (
    SELECT {p} AS patient_id, COUNT(DISTINCT {q(ids["LAB"])}) AS lab_count
    FROM {fqn("LAB")} GROUP BY 1
),
surg AS (
    SELECT {p} AS patient_id,
           COUNT(DISTINCT {q(ids["SURGICAL_HISTORY"])}) AS surgical_history_count
    FROM {fqn("SURGICAL_HISTORY")} GROUP BY 1
),
med AS (
    SELECT {p} AS patient_id,
           COUNT(DISTINCT {q(ids["MEDICAL_HISTORY"])}) AS medical_history_count
    FROM {fqn("MEDICAL_HISTORY")} GROUP BY 1
),
fam AS (
    SELECT {p} AS patient_id,
           COUNT(DISTINCT {q(ids["FAMILY_HISTORY"])}) AS family_history_count
    FROM {fqn("FAMILY_HISTORY")} GROUP BY 1
),
notes AS (
    SELECT {p} AS patient_id, COUNT(DISTINCT {q(ids["CLINICAL_NOTE"])}) AS note_count
    FROM {fqn("CLINICAL_NOTE")} GROUP BY 1
),
claims AS (
    SELECT {p} AS patient_id,
           COUNT(DISTINCT {ck}) AS claim_count
    FROM {fqn("CLAIM")} GROUP BY 1
)
SELECT
    c.patient_id,
    COALESCE(v.visit_count, 0) AS visit_count,
    COALESCE(l.lab_count, 0) AS lab_count,
    COALESCE(s.surgical_history_count, 0) AS surgical_history_count,
    COALESCE(m.medical_history_count, 0) AS medical_history_count,
    COALESCE(f.family_history_count, 0) AS family_history_count,
    COALESCE(n.note_count, 0) AS note_count,
    COALESCE(cl.claim_count, 0) AS claim_count
FROM census c
LEFT JOIN visits v ON v.patient_id = c.patient_id
LEFT JOIN labs l ON l.patient_id = c.patient_id
LEFT JOIN surg s ON s.patient_id = c.patient_id
LEFT JOIN med m ON m.patient_id = c.patient_id
LEFT JOIN fam f ON f.patient_id = c.patient_id
LEFT JOIN notes n ON n.patient_id = c.patient_id
LEFT JOIN claims cl ON cl.patient_id = c.patient_id
'''

patients = session.sql(patient_sql).to_pandas()
patients.columns = [c.lower() for c in patients.columns]
print(f"Patients: {len(patients)}")
display(patients.sort_values(["visit_count", "patient_id"], ascending=[False, True]).head(50))


In [ ]:
# 3) Save today's snapshot
run_id = session.sql("SELECT UUID_STRING() AS RUN_ID").collect()[0]["RUN_ID"]

session.sql(f'''
    INSERT INTO {snap("VOLUME_CHECK_RUN")} (run_id, source_db, source_schema)
    SELECT '{run_id}', '{CONFIG["source_db"]}', '{CONFIG["source_schema"]}'
''').collect()

# write via temp tables so we do not round-trip millions of patient rows through pandas
session.sql("CREATE OR REPLACE TEMP TABLE tmp_table_sizes AS SELECT * FROM (" + " UNION ALL ".join(parts) + ")").collect()
session.sql("CREATE OR REPLACE TEMP TABLE tmp_patient_counts AS " + patient_sql).collect()

session.sql(f'''
    INSERT INTO {snap("VOLUME_TABLE_SNAPSHOT")}
    SELECT '{run_id}', table_name, row_count, distinct_patients, distinct_encounters, distinct_record_ids
    FROM tmp_table_sizes
''').collect()

session.sql(f'''
    INSERT INTO {snap("VOLUME_PATIENT_SNAPSHOT")}
    SELECT '{run_id}', patient_id, visit_count, lab_count, surgical_history_count,
           medical_history_count, family_history_count, note_count,
           claim_count
    FROM tmp_patient_counts
''').collect()

print(f"Saved snapshot run_id={run_id}")


In [ ]:
# 4) Diff vs previous snapshot
runs = session.sql(f'''
    SELECT run_id, checked_at,
           ROW_NUMBER() OVER (ORDER BY checked_at DESC) AS rn
    FROM {snap("VOLUME_CHECK_RUN")}
''').to_pandas()
runs.columns = [c.lower() for c in runs.columns]
display(runs.head(10))

table_diff = session.sql(f'''
    WITH runs AS (
        SELECT run_id, ROW_NUMBER() OVER (ORDER BY checked_at DESC) AS rn
        FROM {snap("VOLUME_CHECK_RUN")}
    ),
    today AS (
        SELECT s.* FROM {snap("VOLUME_TABLE_SNAPSHOT")} s
        JOIN runs r ON r.run_id = s.run_id AND r.rn = 1
    ),
    prev AS (
        SELECT s.* FROM {snap("VOLUME_TABLE_SNAPSHOT")} s
        JOIN runs r ON r.run_id = s.run_id AND r.rn = 2
    )
    SELECT
        t.table_name,
        t.row_count AS today_rows,
        COALESCE(p.row_count, 0) AS prev_rows,
        t.row_count - COALESCE(p.row_count, 0) AS row_delta,
        t.distinct_patients AS today_patients,
        COALESCE(p.distinct_patients, 0) AS prev_patients,
        t.distinct_patients - COALESCE(p.distinct_patients, 0) AS patient_delta,
        t.distinct_encounters AS today_encounters,
        COALESCE(p.distinct_encounters, 0) AS prev_encounters,
        t.distinct_encounters - COALESCE(p.distinct_encounters, 0) AS encounter_delta,
        t.distinct_record_ids AS today_ids,
        COALESCE(p.distinct_record_ids, 0) AS prev_ids,
        t.distinct_record_ids - COALESCE(p.distinct_record_ids, 0) AS id_delta,
        CASE WHEN t.row_count > COALESCE(p.row_count, 0) THEN 'NEW DATA'
             WHEN t.row_count < COALESCE(p.row_count, 0) THEN 'SHRANK'
             ELSE 'NO ROW CHANGE' END AS status
    FROM today t
    LEFT JOIN prev p ON p.table_name = t.table_name
    ORDER BY t.table_name
''').to_pandas()
table_diff.columns = [c.lower() for c in table_diff.columns]
display(table_diff)

if len(runs) < 2:
    print("First snapshot only - run again after the next load to see a day-over-day diff.")
else:
    new_data = (table_diff["row_delta"] > 0).any()
    shrank = (table_diff["row_delta"] < 0).any()
    if new_data:
        print("NEW DATA loaded (at least one table grew).")
    elif shrank:
        print("No growth - at least one table shrank.")
    else:
        print("No row-count change vs previous snapshot.")


In [ ]:
# 5) New patients, and patients whose visits/labs/claims grew
if len(runs) < 2:
    print("Skip patient diff until a second snapshot exists.")
else:
    new_patients = session.sql(f'''
        WITH runs AS (
            SELECT run_id, ROW_NUMBER() OVER (ORDER BY checked_at DESC) AS rn
            FROM {snap("VOLUME_CHECK_RUN")}
        )
        SELECT t.*
        FROM {snap("VOLUME_PATIENT_SNAPSHOT")} t
        JOIN runs r1 ON r1.run_id = t.run_id AND r1.rn = 1
        WHERE NOT EXISTS (
            SELECT 1
            FROM {snap("VOLUME_PATIENT_SNAPSHOT")} p
            JOIN runs r2 ON r2.run_id = p.run_id AND r2.rn = 2
            WHERE p.patient_id = t.patient_id
        )
        ORDER BY t.patient_id
    ''').to_pandas()
    new_patients.columns = [c.lower() for c in new_patients.columns]
    print(f"New patients: {len(new_patients)}")
    display(new_patients.head(50))

    grew = session.sql(f'''
        WITH runs AS (
            SELECT run_id, ROW_NUMBER() OVER (ORDER BY checked_at DESC) AS rn
            FROM {snap("VOLUME_CHECK_RUN")}
        ),
        today AS (
            SELECT s.* FROM {snap("VOLUME_PATIENT_SNAPSHOT")} s
            JOIN runs r ON r.run_id = s.run_id AND r.rn = 1
        ),
        prev AS (
            SELECT s.* FROM {snap("VOLUME_PATIENT_SNAPSHOT")} s
            JOIN runs r ON r.run_id = s.run_id AND r.rn = 2
        )
        SELECT
            t.patient_id,
            t.visit_count - p.visit_count AS visit_delta,
            t.lab_count - p.lab_count AS lab_delta,
            t.surgical_history_count - p.surgical_history_count AS surgical_delta,
            t.medical_history_count - p.medical_history_count AS medical_delta,
            t.family_history_count - p.family_history_count AS family_delta,
            t.note_count - p.note_count AS note_delta,
            t.claim_count - p.claim_count AS claim_delta
        FROM today t
        JOIN prev p ON p.patient_id = t.patient_id
        WHERE t.visit_count > p.visit_count
           OR t.lab_count > p.lab_count
           OR t.surgical_history_count > p.surgical_history_count
           OR t.medical_history_count > p.medical_history_count
           OR t.family_history_count > p.family_history_count
           OR t.note_count > p.note_count
           OR t.claim_count > p.claim_count
        ORDER BY visit_delta DESC, lab_delta DESC, t.patient_id
    ''').to_pandas()
    grew.columns = [c.lower() for c in grew.columns]
    print(f"Patients with more visits/labs/history/notes/claims: {len(grew)}")
    display(grew.head(50))


In [ ]:
print("Done.")
